In [0]:
from delta.tables import DeltaTable

def write_delta_table(
    df,
    table_name: str,
    write_mode: str,
    merge_key: str| list | None = None,
    cluster_keys: list | None = None,
    option_dict: dict | None = None
) -> None:
    default_options = {
        "mergeSchema": "true",
        "tblproperties.delta.autoOptimize.optimizeWrite": "true",
        "tblproperties.delta.autoOptimize.autoCompact": "true",
        "tblproperties.delta.enableChangeDataFeed": "true"
    }

    if option_dict:
        default_options.update(option_dict)

    target_table = spark.catalog.tableExists(table_name)
    if target_table and write_mode.lower() == "merge":
        if not merge_key:
            raise ValueError(f"for merging the Table {table_name} merge_key is required")

        dt = DeltaTable.forName(spark, table_name)

        if isinstance(merge_key, str):
            merge_key =[merge_key]

        merge_condition = "AND".join([f"target.{k} == source{k}" for k in merge_key])
        (
            dt.alias("target")
            .merge(df.alias("source"), merge_condition)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    elif target_table and write_mode.lower() == "append":
        (
            df.write
            .format("delta")
            .mode("append")
            .options(**default_options)
            .saveAsTable(table_name)
        )
    else:
        writer = df.write.format("delta").mode("overwrite")

        if cluster_keys:
            writer = writer.clusterBy(*cluster_keys)

        writer.options(**default_options).saveAsTable(table_name)